# Module 05 — Model-Based RL: Dyna-Q

**Reinforcement Learning · Dakar Institute of Technology (DIT)**

**Model-free** methods (Module 03) throw away each transition after one update.
**Model-based** methods instead **learn a model** of the environment from
experience — $\hat P(s'\mid s,a)$ and $\hat R(s,a)$ — and then **plan** with it:
generate *imagined* experience and learn from that too. This makes each expensive
real interaction go much further (**sample efficiency**).

### Dyna-Q = direct RL + model learning + planning
Every real step does three things (Sutton & Barto, Ch. 8):
1. **Direct RL:** a normal Q-learning update from the real transition.
2. **Model learning:** record `Model[s,a] = (r, s')`.
3. **Planning:** repeat `n` times — sample a previously-seen `(s,a)`, look up the
   model's `(r, s')`, and do a Q-learning update on that *simulated* transition.

With `n = 0` Dyna-Q **is** plain Q-learning. As `n` grows, the agent solves the task
in far fewer *real* steps.

### Learning objectives
1. Implement a tabular **learned model**.
2. Implement **Dyna-Q** and its planning loop.
3. Show that **planning dramatically cuts the number of real episodes** needed.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "rl_helpers").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from rl_helpers import (GridWorld, set_seed, epsilon_greedy, greedy_policy_from_Q,
                        plot_policy, animate_policy)

set_seed(0)
rng = np.random.default_rng(0)

# A deterministic maze (slip=0) — the classic Dyna testbed. Bigger than before,
# with a wall the agent must go around.
maze = GridWorld(
    n_rows=6, n_cols=9,
    start=(2, 0),
    terminals={(0, 8): 1.0},
    walls={(1, 2), (2, 2), (3, 2), (4, 5), (0, 7), (1, 7), (2, 7)},
    step_reward=0.0, slip=0.0, gamma=0.95,
)
print(maze.render_ascii())

## 1. Dyna-Q

We track **steps per episode**: the classic Dyna metric. Fewer steps = a better
path found faster. We'll compare `n_planning ∈ {0, 5, 50}`.

In [ ]:
def dyna_q(env, n_episodes=50, n_planning=5, alpha=0.1, gamma=None, eps=0.1,
           max_steps=2000):
    gamma = env.gamma if gamma is None else gamma
    Q = np.zeros((env.n_states, env.n_actions))
    model = {} # (s, a) -> (r, s', done): the learned model
    seen = [] # visited (s, a) pairs to sample during planning
    steps_per_episode = []
    for _ in range(n_episodes):
        s = env.reset()
        steps = 0
        for _ in range(max_steps):
            a = epsilon_greedy(Q[s], eps, env.n_actions, rng)
            ns, r, done, _ = env.step(a)
            # (1) direct RL update
            # TODO: Q-learning update of Q[s, a] from the REAL transition
            raise NotImplementedError("Direct RL update")
            # (2) model learning
            if (s, a) not in model:
                seen.append((s, a))
            model[(s, a)] = (r, ns, done)
            # (3) planning: repeat n_planning times
            for _ in range(n_planning):
                # TODO: sample a random previously-seen (ps, pa) from `seen`,
                # look up (pr, pns, pdone) in `model`, and do a Q-learning
                # update on that SIMULATED transition.
                raise NotImplementedError("Planning update")
            s = ns
            steps += 1
            if done:
                break
        steps_per_episode.append(steps)
    return Q, steps_per_episode

def average_runs(n_planning, n_runs=10, n_episodes=50):
    all_curves = []
    for run in range(n_runs):
        global rng
        rng = np.random.default_rng(run)
        _, steps = dyna_q(maze, n_episodes=n_episodes, n_planning=n_planning)
        all_curves.append(steps)
    return np.mean(all_curves, axis=0)

fig, ax = plt.subplots(figsize=(8, 5))
for n in [0, 5, 50]:
    curve = average_runs(n)
    ax.plot(range(1, len(curve) + 1), curve, label=f"{n} planning steps", lw=2)
ax.set_xlabel("Episode"); ax.set_ylabel("Steps to reach goal")
ax.set_title("Dyna-Q on a maze: planning slashes the steps per episode")
ax.set_ylim(0, 800); ax.legend(); ax.grid(alpha=0.3)
plt.show()

The `n=0` curve (pure Q-learning) needs many episodes to shorten its path, while
`n=50` finds a near-optimal route after just a handful of real episodes — **the same
real experience, reused by planning**. That is the whole promise of model-based RL.

## 2. Inspect the learned policy & watch it act

In [ ]:
rng = np.random.default_rng(0)
Q_dyna, _ = dyna_q(maze, n_episodes=50, n_planning=50)
policy = greedy_policy_from_Q(Q_dyna)
plot_policy(maze, policy, "Dyna-Q greedy policy (50 planning steps)")
plt.show()
animate_policy(maze, lambda s: int(policy[s]), max_steps=40, fps=4, path="dyna_maze.gif")

## 3. Discussion: why (and when) planning helps

- **Sample efficiency:** planning re-uses real transitions many times, so fewer
  *real* interactions are needed — crucial when real steps are expensive (robots).
- **The catch — a wrong model misleads planning.** If the environment *changes*
  (a shortcut opens, or a path is blocked), a stale model keeps planning against
  the old world. Extensions like **Dyna-Q+** add an *exploration bonus* for
  long-unvisited state–actions to detect such changes.
- **Stochastic environments** need a *distribution* model (counts of outcomes), not
  the single-outcome deterministic model above — see the assignment.

## Recap
- **Model-based RL** learns $\hat P, \hat R$ from experience and **plans** with them.
- **Dyna-Q** interleaves real Q-learning updates with `n` simulated ones per step.
- More planning ⇒ far fewer real episodes to reach a good policy.

### Assignment (see assignment notebook / README)
1. Add a **changing environment** (block the learned shortcut mid-training) and
   compare Dyna-Q vs **Dyna-Q+**.
2. Extend the model to be **stochastic** and run Dyna-Q on a *slippery* GridWorld
   or `FrozenLake-v1`.

**Next:** Module 06 — policy-gradient & actor-critic methods.